In [ ]:
!pip install transformers datasets scikit-learn


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"


In [ ]:
!git clone https://github.com/carabasroxana/psy-detectives.git

fatal: destination path 'psy-detectives' already exists and is not an empty directory.


In [ ]:
!rm -rf semeval26_starter_pack
!git clone https://github.com/carabasroxana/psy-detectives.git

fatal: destination path 'psy-detectives' already exists and is not an empty directory.


In [ ]:
!ls psy-detectives/data


candidate_segments_for_annotation.jsonl  train_redacted.jsonl
documents_for_annotation.jsonl		 train_rehydrated.jsonl
psy_dict.csv


In [ ]:
import json
import pandas as pd
from sklearn.model_selection import train_test_split

DOCS_PATH = "/content/psy-detectives/data/documents_for_annotation.jsonl"
LABELS_PATH = "/content/psy-detectives/data/train_redacted.jsonl"

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

docs_df = load_jsonl(DOCS_PATH)
labs_df = load_jsonl(LABELS_PATH)

print("docs_df:", docs_df.head(2))
print("labs_df:", labs_df.head(2))

# aliniem coloanele de id
docs_df = docs_df.rename(columns={"id": "_id"})
labs_df_small = labs_df[["_id", "conspiracy"]]

# join text + label
df = docs_df.merge(labs_df_small, on="_id", how="inner")
print("Merged:", df.head(3))

# păstrăm doar Yes / No
df = df[df["conspiracy"].isin(["Yes", "No"])].copy()
df["label"] = df["conspiracy"].map({"No": 0, "Yes": 1})
df = df[["text", "label"]]

print("Label counts:\n", df["label"].value_counts())

# împărțire train / dev
train_df, dev_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_df.to_csv("train.csv", index=False)
dev_df.to_csv("dev.csv", index=False)

print("Train size:", len(train_df))
print("Dev size:", len(dev_df))


docs_df:            id                                               text  \
0  t1_f7ju17o  A great article on what's taking place in Boli...   
1  t1_k5c5yyz  Chris Lehto interviews Ashton Forbes about his...   

                                                meta  
0  {'subreddit': 'conspiracy', 'existing_conspira...  
1  {'subreddit': 'HighStrangeness', 'existing_con...  
labs_df:           _id  conspiracy                                            markers  \
0  t1_f7ju17o         Yes  [{'startIndex': 8, 'endIndex': 15, 'type': 'Ev...   
1  t1_k5c5yyz  Can't tell                                                 []   

         subreddit    annotator  
0       conspiracy  annotator_0  
1  HighStrangeness  annotator_1  
Merged:           _id                                               text  \
0  t1_f7ju17o  A great article on what's taking place in Boli...   
1  t1_k5c5yyz  Chris Lehto interviews Ashton Forbes about his...   
2  t1_givys64  Germany has upset other EU member states b

In [ ]:
!pip install transformers datasets scikit-learn


In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import torch

print("Torch:", torch.__version__)

# Load CSV created earlier
train_df = pd.read_csv("train.csv")
dev_df   = pd.read_csv("dev.csv")

print(train_df.head())
print(dev_df.head())

# Convert to HF dataset
train_ds = Dataset.from_pandas(train_df)
dev_ds   = Dataset.from_pandas(dev_df)

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_tok = train_ds.map(tokenize, batched=True)
dev_tok   = dev_ds.map(tokenize, batched=True)

train_tok = train_tok.rename_column("label", "labels")
dev_tok   = dev_tok.rename_column("label", "labels")

train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
dev_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
        "precision": precision_score(labels, preds, average="weighted"),
        "recall": recall_score(labels, preds, average="weighted"),
    }

training_args = TrainingArguments(
    output_dir="./baseline",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=dev_tok,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)




Torch: 2.9.0+cpu
                                                text  label
0  Recently on reddit someone came forward to exp...      0
1  Oct 8, 2019 - The governor of the Bank of Engl...      0
2  Feb 14, 2021 - “The trial record was a complet...      1
3  Saw this video and really thought you guys wou...      0
4  Many have wondered what the impacts of the gre...      0
                                                text  label
0  Another intercept informant caught and is bein...      1
1  Hello world! This is a video I made summarizin...      1
2  Just because nobody understands how a medicine...      1
3  Welcome to the demagogue anarchy antidote, glo...      1
4  RMFN, since I know you read this subreddit, kn...      1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/3822 [00:00<?, ? examples/s]

Map:   0%|          | 0/956 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-1666063405.py:67: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

eval_results = trainer.evaluate()
eval_results

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.652600
100,0.629400
150,0.600400
200,0.586900
250,0.548900
300,0.465000
350,0.527600
400,0.489300
450,0.447500
500,0.477100


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.5440186262130737,
 'eval_accuracy': 0.7625523012552301,
 'eval_f1': 0.7605383906431893,
 'eval_precision': 0.7619993593500685,
 'eval_recall': 0.7625523012552301,
 'eval_runtime': 343.1884,
 'eval_samples_per_second': 2.786,
 'eval_steps_per_second': 0.087,
 'epoch': 3.0}

In [ ]:
!git clone https://github.com/carabasroxana/psy-detectives.git


Cloning into 'psy-detectives'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 72 (delta 24), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 2.13 MiB | 4.37 MiB/s, done.
Resolving deltas: 100% (24/24), done.


In [ ]:
import json
import pandas as pd
from datasets import Dataset

# Load test documents
DOCS_PATH = "psy-detectives/data/documents_for_annotation.jsonl"

def load_jsonl_df(path):
    rows = []
    with open(path, "r", encoding="utf8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

test_docs = load_jsonl_df(DOCS_PATH)

print("Columns:", test_docs.columns)
print(test_docs.head(2))

# normalize id column
if "id" in test_docs.columns and "_id" not in test_docs.columns:
    test_docs = test_docs.rename(columns={"id": "_id"})

assert "_id" in test_docs.columns
assert "text" in test_docs.columns

test_ds = Dataset.from_pandas(test_docs[["_id", "text"]])


Columns: Index(['id', 'text', 'meta'], dtype='object')
           id                                               text  \
0  t1_f7ju17o  A great article on what's taking place in Boli...   
1  t1_k5c5yyz  Chris Lehto interviews Ashton Forbes about his...   

                                                meta  
0  {'subreddit': 'conspiracy', 'existing_conspira...  
1  {'subreddit': 'HighStrangeness', 'existing_con...  


In [ ]:
from datasets import Dataset
import numpy as np


# dataset pentru inferență
test_ds = Dataset.from_pandas(test_docs[["_id", "text"]])

def tokenize_test(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

test_tok = test_ds.map(tokenize_test, batched=True)
test_tok.set_format(type="torch", columns=["input_ids", "attention_mask"])

pred_out = trainer.predict(test_tok)
preds = pred_out.predictions.argmax(axis=-1)  # 0/1


Map:   0%|          | 0/4316 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
import json

label_map = {0: "No", 1: "Yes"}

with open("submission.jsonl", "w", encoding="utf-8") as f:
    for _id, p in zip(test_docs["_id"].tolist(), preds):
        f.write(
            json.dumps(
                {"_id": _id, "conspiracy": label_map[int(p)]},
                ensure_ascii=False
            ) + "\n"
        )

print("submission.jsonl written")
!head submission.jsonl


submission.jsonl written
{"_id": "t1_f7ju17o", "conspiracy": "No"}
{"_id": "t1_k5c5yyz", "conspiracy": "No"}
{"_id": "t1_givys64", "conspiracy": "No"}
{"_id": "t1_joq538t", "conspiracy": "No"}
{"_id": "t1_hlk1vci", "conspiracy": "No"}
{"_id": "t1_cguw8w8", "conspiracy": "No"}
{"_id": "t1_fmcunuq", "conspiracy": "No"}
{"_id": "t1_jolyktt", "conspiracy": "No"}
{"_id": "t1_fegajpj", "conspiracy": "No"}
{"_id": "t1_k4a1rz6", "conspiracy": "No"}


In [ ]:
from google.colab import files
files.download("submission.jsonl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!cp track2_mcq.jsonl track1_saq.jsonl
!head track1_saq.jsonl

!zip submission_track1.zip track1_saq.jsonl
!unzip -l submission_track1.zip

from google.colab import files
files.download("submission_track1.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json

inp = "track2_mcq.jsonl"   # fișierul tău existent
out = "track1_saq.jsonl"

with open(inp, "r", encoding="utf-8") as fin, open(out, "w", encoding="utf-8") as fout:
    for line in fin:
        if not line.strip():
            continue
        obj = json.loads(line)

        # id poate fi _id sau id
        ex_id = obj.get("_id", obj.get("id"))
        pred  = obj.get("conspiracy", obj.get("label", obj.get("answer")))

        # scriem cu chei multiple (ca să acoperim ce caută evaluatorul)
        new_obj = {
            "id": ex_id,
            "_id": ex_id,
            "label": pred,
            "answer": pred,
            "conspiracy": pred
        }
        fout.write(json.dumps(new_obj, ensure_ascii=False) + "\n")

print("Wrote", out)
!head -n 3 track1_saq.jsonl


Wrote track1_saq.jsonl
{"id": "t1_f7ju17o", "_id": "t1_f7ju17o", "label": "No", "answer": "No", "conspiracy": "No"}
{"id": "t1_k5c5yyz", "_id": "t1_k5c5yyz", "label": "No", "answer": "No", "conspiracy": "No"}
{"id": "t1_givys64", "_id": "t1_givys64", "label": "No", "answer": "No", "conspiracy": "No"}


In [ ]:
!zip -j submission_track1.zip track1_saq.jsonl
!unzip -l submission_track1.zip

from google.colab import files
files.download("submission_track1.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!ls


baseline  psy-detectives  submission_track2.zip  track2_mcq.jsonl
dev.csv   sample_data	  submission.zip	 train.csv
